In [22]:
import pandas as pd
nav = pd.read_csv("../data/raw/02_nav_history.csv")
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(["amfi_code", "date"])
full_dates = pd.date_range(nav["date"].min(), nav["date"].max(), freq="D")

def complete_nav_history(group):
    amfi_code = group.name
    completed = group.set_index("date")[["nav"]].reindex(full_dates).ffill()
    completed["amfi_code"] = amfi_code
    return completed.reset_index(names="date")

nav = (
    nav.groupby("amfi_code", group_keys=False)
    .apply(complete_nav_history)
    .reset_index(drop=True)
    [["amfi_code", "date", "nav"]]
)

nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [7]:
nav.groupby("amfi_code").size().describe()

count      40.0
mean     1608.0
std         0.0
min      1608.0
25%      1608.0
50%      1608.0
75%      1608.0
max      1608.0
dtype: float64

In [8]:
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
print(transactions.shape)
print(transactions["transaction_type"].value_counts())
print(transactions["kyc_status"].value_counts())

(32778, 13)
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [9]:
print("Missing values:", transactions.isna().sum())
print("Duplicates:", transactions.duplicated().sum())
print("Rows:", len(transactions))
print("Investor IDs:", transactions["investor_id"].nunique())

Missing values: investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64
Duplicates: 0
Rows: 32778
Investor IDs: 5000


In [10]:
nav.to_csv("../data/processed/02_nav_history_clean.csv", index=False)
import os

os.path.exists("../data/processed/02_nav_history_clean.csv")

True

In [11]:
print(transactions["amount_inr"].isna().sum())
print((transactions["amount_inr"] <= 0).sum())
print(transactions["transaction_date"].isna().sum())
print(transactions["transaction_date"].min())
print(transactions["transaction_date"].max())

0
0
0
2024-01-01
2025-05-30


In [12]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"],errors="coerce")
print(transactions["transaction_date"].isna().sum())
print(transactions.dtypes)

0
investor_id                      str
transaction_date      datetime64[us]
amfi_code                      int64
transaction_type                 str
amount_inr                     int64
state                            str
city                             str
city_tier                        str
age_group                        str
gender                           str
annual_income_lakh           float64
payment_mode                     str
kyc_status                       str
dtype: object


In [13]:
print("Missing values:")
print(transactions.isna().sum())
print("Duplicates:", transactions.duplicated().sum())
print("Invalid amounts:", (transactions["amount_inr"] <= 0).sum())
print("Transaction types:")
print(transactions["transaction_type"].unique())
print("KYC statuses:")
print(transactions["kyc_status"].unique())
print("Rows:", len(transactions))

Missing values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64
Duplicates: 0
Invalid amounts: 0
Transaction types:
<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
KYC statuses:
<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str
Rows: 32778


In [14]:
transactions.to_csv("../data/processed/08_investor_transactions_clean.csv", index=False)

import os
os.path.exists("../data/processed/08_investor_transactions_clean.csv")

True

In [15]:
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
print(performance.shape)
print(performance.dtypes)

(40, 19)
amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
scheme_aum_crore        int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object


In [16]:
print("Missing values:")
print(performance.isna().sum())
print("Negative scheme AUM:", (performance["scheme_aum_crore"] < 0).sum())
print("Invalid expense ratios:", ((performance["expense_ratio_pct"] < 0.1) | (performance["expense_ratio_pct"] > 2.5)).sum())

Missing values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
scheme_aum_crore      0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64
Negative scheme AUM: 0
Invalid expense ratios: 0


In [17]:
print(performance[[
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct"
]].describe())

       return_1yr_pct  return_3yr_pct  return_5yr_pct  benchmark_3yr_pct  \
count       40.000000       40.000000       40.000000          40.000000   
mean        14.376000       14.089000       14.516750          12.835500   
std          4.883023        4.617253        4.454021           4.740972   
min          4.260000        5.140000        5.430000           3.960000   
25%         11.735000       12.035000       12.340000          10.690000   
50%         14.620000       14.205000       14.185000          13.090000   
75%         16.392500       15.882500       17.585000          14.775000   
max         24.930000       23.390000       23.800000          22.160000   

           alpha       beta  sharpe_ratio  sortino_ratio  std_dev_ann_pct  \
count  40.000000  40.000000     40.000000      40.000000        40.000000   
mean    1.253500   0.873250      1.361750       2.082500        14.962500   
std     0.447412   0.224846      1.475805       2.203144         6.669282   
min    

In [18]:
print("Sharpe ratio > 3:")
print(performance[performance["sharpe_ratio"] > 3][["scheme_name", "sharpe_ratio"]])
print("Sortino ratio > 5:")
print(performance[performance["sortino_ratio"] > 5][["scheme_name", "sortino_ratio"]])

Sharpe ratio > 3:
                                 scheme_name  sharpe_ratio
14  ICICI Pru Liquid Fund - Regular - Growth          7.68
23      Kotak Liquid Fund - Regular - Growth          6.18
30       ABSL Liquid Fund - Regular - Growth          5.14
Sortino ratio > 5:
                                 scheme_name  sortino_ratio
14  ICICI Pru Liquid Fund - Regular - Growth          10.37
23      Kotak Liquid Fund - Regular - Growth           9.70
30       ABSL Liquid Fund - Regular - Growth           8.76


In [19]:
performance["anomaly_flag"] = (
    (performance["sharpe_ratio"] > 3) |
    (performance["sortino_ratio"] > 5)
)
print(performance["anomaly_flag"].value_counts())

anomaly_flag
False    37
True      3
Name: count, dtype: int64


In [20]:
performance.to_csv("../data/processed/07_scheme_performance_clean.csv", index=False)

import os
os.path.exists("../data/processed/07_scheme_performance_clean.csv")

True

In [21]:
from pathlib import Path
from sqlalchemy import create_engine, inspect

engine = create_engine(f"sqlite:///{Path('../bluestock_mf.db').resolve()}")

expected_tables = [
    "dim_fund",
    "dim_date",
    "dim_fund_house",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum",
    "fact_sip",
    "fact_folio_count",
    "fact_category_inflows",
    "fact_benchmark",
]

existing_tables = set(inspect(engine).get_table_names())
for table in expected_tables:
    if table in existing_tables:
        with engine.connect() as conn:
            count = conn.exec_driver_sql(f"SELECT COUNT(*) FROM {table}").scalar()
        print(table, count)
    else:
        print(table, "missing; run load_database.py first")

dim_fund missing; run load_database.py first
dim_date missing; run load_database.py first
dim_fund_house missing; run load_database.py first
fact_nav missing; run load_database.py first
fact_transactions missing; run load_database.py first
fact_performance missing; run load_database.py first
fact_aum missing; run load_database.py first
fact_sip missing; run load_database.py first
fact_folio_count missing; run load_database.py first
fact_category_inflows missing; run load_database.py first
fact_benchmark missing; run load_database.py first
